creating a composite synthetic index from all the supporting data using chow-lin algorithm

In [1]:
import pandas as pd
import numpy as np

In [2]:
# BUILD THE MONTHLY MASTER GRID
# Load Local Monthly Variables (Country-Specific)
df_exrate = pd.read_csv('exchange_rates_cleaned.csv')
df_cpi = pd.read_csv('CPI_melted.csv') 
df_trade = pd.read_csv('international_trade_in_Goods_cleaned.csv')
print(df_exrate.head())
print(df_cpi.head())
print(df_trade.head())

                            COUNTRY        Date  Exchange_Rate
0  Afghanistan, Islamic Republic of  2000-01-01      46.791100
1  Afghanistan, Islamic Republic of  2000-01-01      46.795800
2  Afghanistan, Islamic Republic of  2000-02-01      47.505938
3  Afghanistan, Islamic Republic of  2000-02-01      47.504800
4  Afghanistan, Islamic Republic of  2000-03-01      47.267200
                            COUNTRY  \
0  Afghanistan, Islamic Republic of   
1  Afghanistan, Islamic Republic of   
2  Afghanistan, Islamic Republic of   
3  Afghanistan, Islamic Republic of   
4  Afghanistan, Islamic Republic of   

                                  COICOP_1999                  INDEX_TYPE  \
0  Alcoholic beverages, tobacco and narcotics  Consumer price index (CPI)   
1  Alcoholic beverages, tobacco and narcotics  Consumer price index (CPI)   
2                               Communication  Consumer price index (CPI)   
3                       Clothing and footwear  Consumer price index (CPI)   
4 

In [3]:
# Load Global Monthly Variables (Macro Gravity)
df_fed = pd.read_csv('FEDFUNDS_Final_Cleaned.csv')
df_vix = pd.read_csv('VIX_Final_Cleaned.csv')
df_dxy = pd.read_csv('DXY_Final_Cleaned.csv')
df_brent = pd.read_csv('Global_price_of_Brent_Crude.csv')
print(df_fed.head())
print(df_vix.head())
print(df_dxy.head())
print(df_brent.head())

   FEDFUNDS        Date
0      5.45  2000-01-01
1      5.73  2000-02-01
2      5.85  2000-03-01
3      6.02  2000-04-01
4      6.27  2000-05-01
         Date  Monthly_Avg_VIXCLS
0  2000-01-01           23.202000
1  2000-02-01           23.595500
2  2000-03-01           22.718261
3  2000-04-01           27.164211
4  2000-05-01           26.373182
         Date   DXY_Index
0  2006-01-01  100.000005
1  2006-02-01  100.211170
2  2006-03-01  100.428087
3  2006-04-01   99.743480
4  2006-05-01   97.511774
         Date  Crude_Oil_Price
0  2000-01-01        25.633333
1  2000-02-01        28.030476
2  2000-03-01        27.494348
3  2000-04-01        23.153500
4  2000-05-01        27.805217


In [4]:
# 1. Rename CPI's time column to match the rest of dtaframes for merging 
if 'MONTH_YEAR' in df_cpi.columns:
    df_cpi = df_cpi.rename(columns={'MONTH_YEAR': 'Date'})
print(df_cpi.head())

                            COUNTRY  \
0  Afghanistan, Islamic Republic of   
1  Afghanistan, Islamic Republic of   
2  Afghanistan, Islamic Republic of   
3  Afghanistan, Islamic Republic of   
4  Afghanistan, Islamic Republic of   

                                  COICOP_1999                  INDEX_TYPE  \
0  Alcoholic beverages, tobacco and narcotics  Consumer price index (CPI)   
1  Alcoholic beverages, tobacco and narcotics  Consumer price index (CPI)   
2                               Communication  Consumer price index (CPI)   
3                       Clothing and footwear  Consumer price index (CPI)   
4                               Communication  Consumer price index (CPI)   

         Date  CPI_VALUE  
0  2000-01-01  61.138141  
1  2000-01-01  61.138141  
2  2000-01-01  61.138141  
3  2000-01-01  61.138141  
4  2000-01-01  61.138141  


In [5]:
# Forcing every dataset to the exact same Datetime format
datasets = [df_exrate, df_cpi, df_trade, df_fed, df_vix, df_dxy, df_brent]
for d in datasets:
    if 'DATE' in d.columns: 
        d.rename(columns={'DATE': 'Date'}, inplace=True)
    # This translates text into actual math-based Time
    d['Date'] = pd.to_datetime(d['Date'])

In [6]:
# 2. Crush Exchange Rates (Take the average if there are multiple rates for one month)
df_exrate = df_exrate.groupby(['COUNTRY', 'Date'], as_index=False)['Exchange_Rate'].mean()
print(df_exrate.head())

                            COUNTRY       Date  Exchange_Rate
0  Afghanistan, Islamic Republic of 2000-01-01      46.793450
1  Afghanistan, Islamic Republic of 2000-02-01      47.505369
2  Afghanistan, Islamic Republic of 2000-03-01      47.267200
3  Afghanistan, Islamic Republic of 2000-04-01      47.267200
4  Afghanistan, Islamic Republic of 2000-05-01      47.267200


In [7]:
# 3. Crush CPI (Take the average across all product categories to get the National CPI)
df_cpi = df_cpi.groupby(['COUNTRY', 'Date'], as_index=False)['CPI_VALUE'].mean()
print(df_cpi.head())

                            COUNTRY       Date  CPI_VALUE
0  Afghanistan, Islamic Republic of 2000-01-01  61.138141
1  Afghanistan, Islamic Republic of 2000-02-01  61.138141
2  Afghanistan, Islamic Republic of 2000-03-01  61.138141
3  Afghanistan, Islamic Republic of 2000-04-01  61.138141
4  Afghanistan, Islamic Republic of 2000-05-01  61.138141


In [8]:
# 4. Flatten Trade Data (Pivot Exports/Imports into their own separate X-variables)
if 'INDICATOR' in df_trade.columns:
    df_trade = df_trade.pivot_table(
        index=['COUNTRY', 'Date'], 
        columns='INDICATOR', 
        values='Trade_in_Goods',
        aggfunc='mean'
    ).reset_index()
    df_trade.columns.name = None # Remove pivot name
print(df_trade.head())

   COUNTRY       Date  Export price index (EPI)  Exports of goods  \
0  Albania 2000-01-01                       NaN          8.503182   
1  Albania 2000-02-01                       NaN          8.503182   
2  Albania 2000-03-01                       NaN          8.503182   
3  Albania 2000-04-01                       NaN          8.503182   
4  Albania 2000-05-01                       NaN          8.503182   

   Exports of goods, Price deflator  Exports of goods, Volume index  \
0                               NaN                             NaN   
1                               NaN                             NaN   
2                               NaN                             NaN   
3                               NaN                             NaN   
4                               NaN                             NaN   

   Import price index  Imports of goods  Imports of goods, Price deflator  \
0                 NaN         11.517991                               NaN   
1   

In [9]:
# MASTER MERGE - taking the exchange rate dataframe as the base and merging in the rest of the variables
df_monthly = df_exrate.copy() # giving it a new name for clarity

In [10]:
# Merge Local Country Data 
df_monthly = pd.merge(df_monthly, df_cpi, on=['COUNTRY', 'Date'], how='left')
df_monthly = pd.merge(df_monthly, df_trade, on=['COUNTRY', 'Date'], how='left')
print(df_monthly.head())

                            COUNTRY       Date  Exchange_Rate  CPI_VALUE  \
0  Afghanistan, Islamic Republic of 2000-01-01      46.793450  61.138141   
1  Afghanistan, Islamic Republic of 2000-02-01      47.505369  61.138141   
2  Afghanistan, Islamic Republic of 2000-03-01      47.267200  61.138141   
3  Afghanistan, Islamic Republic of 2000-04-01      47.267200  61.138141   
4  Afghanistan, Islamic Republic of 2000-05-01      47.267200  61.138141   

   Export price index (EPI)  Exports of goods  \
0                       NaN               NaN   
1                       NaN               NaN   
2                       NaN               NaN   
3                       NaN               NaN   
4                       NaN               NaN   

   Exports of goods, Price deflator  Exports of goods, Volume index  \
0                               NaN                             NaN   
1                               NaN                             NaN   
2                               NaN

In [11]:
# Merge Global Data (Matches ONLY on Date, broadcasting to all countries)
df_monthly = pd.merge(df_monthly, df_fed, on='Date', how='left')
df_monthly = pd.merge(df_monthly, df_vix, on='Date', how='left')
df_monthly = pd.merge(df_monthly, df_dxy, on='Date', how='left')
df_monthly = pd.merge(df_monthly, df_brent, on='Date', how='left')

In [12]:
print("Final Shape:", df_monthly.shape)
display(df_monthly.head())

Final Shape: (63945, 16)


,COUNTRY,Date,Exchange_Rate,CPI_VALUE,Export price index (EPI),Exports of goods,"Exports of goods, Price deflator","Exports of goods, Volume index",Import price index,Imports of goods,"Imports of goods, Price deflator","Imports of goods, Volume index",FEDFUNDS,Monthly_Avg_VIXCLS,DXY_Index,Crude_Oil_Price
0,"Afghanistan, Islamic Republic of",2000-01-01,46.793450,61.138141,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.45,23.202000,NaN,25.633333
1,"Afghanistan, Islamic Republic of",2000-02-01,47.505369,61.138141,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.73,23.595500,NaN,28.030476
2,"Afghanistan, Islamic Republic of",2000-03-01,47.267200,61.138141,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.85,22.718261,NaN,27.494348
3,"Afghanistan, Islamic Republic of",2000-04-01,47.267200,61.138141,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.02,27.164211,NaN,23.153500
4,"Afghanistan, Islamic Republic of",2000-05-01,47.267200,61.138141,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.27,26.373182,NaN,27.805217


In [13]:
df_monthly.to_csv('Master_Monthly_Grid.csv', index=False)

STEP 2 & 3: PCA SYNTHETIC INDEX & CHOW-LIN TRIANGULATION

In [14]:
import pip
!pip install tempdisagg

In [15]:
import os
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [16]:
#Inject R's core pathways into the Windows Environment
R_HOME = r"C:\Program Files\R\R-4.6.0"
os.environ['R_HOME'] = R_HOME
# Add the 'bin\x64' folder to the system PATH so stats.dll can find its dependencies
os.environ['PATH'] = R_HOME + r"\bin\x64;" + os.environ.get('PATH', '')

In [17]:
# NOW it is safe to import the rpy2 bridge
import rpy2.robjects as ro
from rpy2.robjects import FloatVector
from rpy2.robjects.packages import importr
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter

In [18]:
# Verify the package loads without crashing
try:
    tempdisagg_r = importr('tempdisagg')
    print("✅ R Environment and tempdisagg loaded successfully!")
except Exception as e:
    print("Error: R package 'tempdisagg' still not found.")
    raise e

✅ R Environment and tempdisagg loaded successfully!


In [19]:
df_nfa = pd.read_csv('NFA_Annual_Anchors_Cleaned.csv')
print(df_nfa.head())


       COUNTRY        Date  Assets, Claims on Central Government (CBS)  \
0  Afghanistan  2006-12-31                                 2003.882126   
1  Afghanistan  2007-12-31                                16706.636211   
2  Afghanistan  2008-12-31                                19227.020024   
3  Afghanistan  2009-12-31                                13650.763048   
4  Afghanistan  2010-12-31                                13650.763048   

   Assets, Claims on Nonresidents (CBS)  \
0                         100285.840640   
1                         133299.139871   
2                         161734.421728   
3                         205060.860621   
4                         226880.956341   

   Assets, Claims on Other depository corporations (CBS)  \
0                                               0.00       
1                                               0.00       
2                                               0.00       
3                                               0.00    

In [20]:
# =================================================================
# THE ENGINE: PCA SYNTHETIC INDEX & OFFICIAL R CHOW-LIN (via rpy2)
# =================================================================
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import rpy2.robjects as ro
from rpy2.robjects import FloatVector
from rpy2.robjects.conversion import localconverter
from rpy2.robjects import pandas2ri

# -----------------------------------------------------------------
# Remind Pandas that 'Date' is a mathematical Time object
# -----------------------------------------------------------------
df_nfa['Date'] = pd.to_datetime(df_nfa['Date'])
df_monthly['Date'] = pd.to_datetime(df_monthly['Date'])

def create_index_and_disaggregate(country, df_annual, df_monthly):
    # 1. Extract Annual NFA Anchor
    nfa_col = 'Net (assets minus liabilities), Net foreign assets (CBS)'
    
    nfa_a = df_annual[df_annual['COUNTRY'] == country].set_index('Date')[nfa_col].dropna()
    
    # 2. Extract Features for the Synthetic Index (NO EXCHANGE RATES)
    features = ['Monthly_Avg_VIXCLS', 'FEDFUNDS', 'CPI_VALUE'] 
    country_monthly = df_monthly[df_monthly['COUNTRY'] == country].set_index('Date').dropna(subset=features)
    
    if len(nfa_a) < 3 or len(country_monthly) < 24:
        return None 
        
    # 3. ENFORCE THE 2021 BOUNDARY (Neutralize Look-Ahead Bias)
    nfa_train = nfa_a[nfa_a.index.year <= 2021]
    monthly_train = country_monthly[country_monthly.index.year <= 2021]
    
    if len(nfa_train) < 3:
        return None

    try:
        # 4. ALIGN THE MATHEMATICAL GRID 
        start_year = max(nfa_train.index.min().year, monthly_train.index.min().year)
        end_year = min(nfa_train.index.max().year, monthly_train.index.max().year)
        
        nfa_train = nfa_train[(nfa_train.index.year >= start_year) & (nfa_train.index.year <= end_year)]
        perfect_months = pd.date_range(start=f"{start_year}-01-01", end=f"{end_year}-12-01", freq='MS')
        monthly_train = monthly_train.reindex(perfect_months).ffill().bfill()

        # 5. CREATE THE SYNTHETIC MACRO INDEX (PCA)
        scaler = StandardScaler()
        scaled_features = scaler.fit_transform(monthly_train[features])
        pca = PCA(n_components=1)
        synthetic_index = pca.fit_transform(scaled_features).flatten()
        
        # 6. THE R-BRIDGE: SEND DATA TO R FOR CHOW-LIN
        with localconverter(ro.default_converter + pandas2ri.converter):
            ro.globalenv['Y_py'] = FloatVector(nfa_train.values)
            ro.globalenv['X_py'] = FloatVector(synthetic_index)
            ro.globalenv['start_yr'] = start_year
            
            # THE FIX: Explicitly forcing R to load the libraries inside the bridge
            r_script = """
                library(stats)
                library(tempdisagg)
                Y_ts <- ts(Y_py, start=c(start_yr, 1), frequency=1)
                X_ts <- ts(X_py, start=c(start_yr, 1), frequency=12)
                model <- td(Y_ts ~ X_ts, conversion="average", method="chow-lin-maxlog")
                as.numeric(predict(model))
            """
            nfa_monthly_train = ro.r(r_script)
        
        # 7. Format the R output back into a Pandas DataFrame
        temp_df = pd.DataFrame({
            'Date': perfect_months, 
            'NFA_Triangulated': np.array(nfa_monthly_train),
            'COUNTRY': country
        })
        return temp_df
    
    except Exception as e:
        # Silently skip countries where the math fails (e.g., singular matrices)
        return None

# =================================================================
# EXECUTE THE LOOP ACROSS ALL COUNTRIES
# =================================================================
print("🧠 Building Synthetic Indices & Routing to R for Chow-Lin... (This may take 30-60 seconds)")
triangulated_data = []
countries = df_monthly['COUNTRY'].unique()

for country in countries:
    df_tri = create_index_and_disaggregate(country, df_nfa, df_monthly) 
    if df_tri is not None:
        triangulated_data.append(df_tri)

# Combine everything into one final DataFrame
df_nfa_all = pd.concat(triangulated_data, ignore_index=True)

print("✅ PCA Synthetic Index & Chow-Lin Triangulation Complete!")
display(df_nfa_all.head(10))

🧠 Building Synthetic Indices & Routing to R for Chow-Lin... (This may take 30-60 seconds)
✅ PCA Synthetic Index & Chow-Lin Triangulation Complete!


,Date,NFA_Triangulated,COUNTRY
0,2001-01-01,81705.744413,Albania
1,2001-02-01,78525.474605,Albania
2,2001-03-01,78759.299676,Albania
3,2001-04-01,83495.331081,Albania
4,2001-05-01,84259.565081,Albania
5,2001-06-01,82900.465699,Albania
6,2001-07-01,87157.948325,Albania
7,2001-08-01,90354.393153,Albania
8,2001-09-01,92874.267767,Albania
9,2001-10-01,92623.245728,Albania


In [21]:
df_nfa_all.to_csv('NFA_Monthly_Triangulated.csv', index=False)